In [1]:
"""******************************************************************************************
* HumARConoid-Sim2Real
*
* Sim2Real Transfer to Optimize Humanoid Locomotion Strategy using Reinforcement Learning
*
*     https://github.com/S-CHOI-S/HumARConoid-Sim2Real.git
*
* Advanced Robot Control Lab. (ARC)
* 	  @ Korea Institute of Science and Technology
*
*	  https://sites.google.com/view/kist-arc
*
******************************************************************************************"""

"* Authors: Sol Choi (Jennifer) *"

import torch

In [ ]:
import os
import copy

class _OnnxPolicyExporter(torch.nn.Module):
    """Exporter of actor-critic into ONNX file."""

    def __init__(self, policy, normalizer=None, verbose=False):
        super().__init__()
        self.verbose = verbose
        self.is_recurrent = policy.is_recurrent
        # copy policy parameters
        if hasattr(policy, "actor"):
            self.actor = copy.deepcopy(policy.actor)
            if self.is_recurrent:
                self.rnn = copy.deepcopy(policy.memory)
        elif hasattr(policy, "student"):
            self.actor = copy.deepcopy(policy.student)
            if self.is_recurrent:
                self.rnn = copy.deepcopy(policy.memory_s.rnn)
        else:
            raise ValueError("Policy does not have an actor/student module.")
        # set up recurrent network
        if self.is_recurrent:
            self.rnn.cpu()
            self.forward = self.forward_lstm
        # copy normalizer if exists
        if normalizer:
            self.normalizer = copy.deepcopy(normalizer)
        else:
            self.normalizer = torch.nn.Identity()

    def forward_lstm(self, x_in, h_in, c_in):
        x_in = self.normalizer(x_in)
        x, (h, c) = self.rnn(x_in.unsqueeze(0), (h_in, c_in))
        x = x.squeeze(0)
        return self.actor(x), h, c

    def forward(self, x):
        return self.actor(self.normalizer(x))

    def export(self, path, filename):
        self.to("cpu")
        if self.is_recurrent:
            obs = torch.zeros(1, self.rnn.input_size)
            h_in = torch.zeros(self.rnn.num_layers, 1, self.rnn.hidden_size)
            c_in = torch.zeros(self.rnn.num_layers, 1, self.rnn.hidden_size)
            actions, h_out, c_out = self(obs, h_in, c_in)
            torch.onnx.export(
                self,
                (obs, h_in, c_in),
                os.path.join(path, filename),
                export_params=True,
                opset_version=11,
                verbose=self.verbose,
                input_names=["obs", "h_in", "c_in"],
                output_names=["actions", "h_out", "c_out"],
                dynamic_axes={},
            )
        else:
            obs = torch.zeros(1, self.actor[0].in_features)
            torch.onnx.export(
                self,
                obs,
                os.path.join(path, filename),
                export_params=True,
                opset_version=11,
                verbose=self.verbose,
                input_names=["obs"],
                output_names=["actions"],
                dynamic_axes={},
            )
            
# policy_exporter = _OnnxPolicyExporter(policy, normalizer, verbose)
model = torch.jit.load("./policy/g1_pretrained/motion.pt")
policy_exporter=_OnnxPolicyExporter(model,None,True)

# policy_exporter.export('.', 'policy.onnx')

In [ ]:
model = torch.jit.load("./policy/g1_pretrained/motion.pt")
model.eval()
print(model)
print(model.graph)
print(model._c._get_method('forward').schema)


dummy_input = torch.randn(1, 47)


# torch.onnx.export(
#   model,
#   dummy_input,
#   "./policy/g1_pretrained/motion_re.onnx",
#   export_params=True,
#   opset_version=16,
#   do_constant_folding=True,
#   input_names=["state"],
#   output_names=["action"],
#   dynamic_axes={
#       "state": {0: "batch_size"},
#       "action": {0: "batch_size"}
#   }
# )

obs = torch.zeros(1, 47)
model = torch.zeros(1, 1, 64)
c_in = torch.zeros(1, 1, 64)
actions, h_out, c_out = (obs, h_in, c_in)

torch.onnx.export(
    model,
    (obs),
    "./policy/g1_pretrained/motion_re.onnx",
    export_params=True,
    opset_version=11,
    input_names=["obs"],
    output_names=["actions"],
    dynamic_axes={},
)

In [ ]:
import onnx

onnx_model = onnx.load("./policy/g1_pretrained/motion_16.onnx")

onnx.checker.check_model(onnx_model)

print("ONNX model verified!")
